# D-FINE-cpp Python quickstart

This notebook uses the latest published release, v0.3.3. The wheel bundles the native
runtime (`libdfine.so`) and engine builder; the release supplies the ONNX artifact. It runs
from a fresh environment without a repository checkout, OpenCV, or a compiler.

Requirements: Linux x86_64, CUDA 12, TensorRT 10.13, and an NVIDIA GPU. The published wheel
is validated on Ada and Blackwell. Turing, Ampere, Jetson, and other targets require a
source build; the remaining steps are unchanged.

In [ ]:
%pip install "dfine[cli,tensorrt] @ https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.3/dfine-0.3.3-py3-none-linux_x86_64.whl"

## 1. Get the production ONNX and compile the engine

TensorRT engines are target-local, so this step compiles the published D-FINE-M slim ONNX
for the installed GPU and TensorRT runtime. The slim graph matches the FP32 reference on the
recorded full-validation COCO gate.

In [ ]:
!curl -fLO https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.3/dfine_m_slim.onnx \
       -fLO https://github.com/PogChamper/dfine-cpp/releases/download/v0.3.3/dfine_m_slim.json
!dfine build --model m --onnx dfine_m_slim.onnx --output dfine_m_slim.engine

## 2. Detect

In [ ]:
import urllib.request

import numpy as np
from PIL import Image

from dfine import Detector

url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats on a couch
img = Image.open(urllib.request.urlopen(url)).convert("RGB")
frame = np.asarray(img)                       # HWC uint8, RGB

det = Detector("dfine_m_slim.engine", threshold=0.5)
detections = det.detect(frame)
for d in detections:
    print(f"{d.class_name:12s} {d.score:.3f}  {tuple(round(v, 1) for v in d.box.as_tuple())}")

In [ ]:
from PIL import ImageDraw

vis = img.copy()
draw = ImageDraw.Draw(vis)
for d in detections:
    x1, y1, x2, y2 = d.box.as_tuple()
    draw.rectangle([x1, y1, x2, y2], outline=(255, 80, 80), width=3)
    draw.text((x1 + 3, max(0, y1 - 12)), f"{d.class_name} {d.score:.2f}", fill=(255, 80, 80))
vis

## 3. How fast is it here?

The loop below measures the installed Python path: CUDA preprocessing, TensorRT execution,
decode, and Python call overhead. The released slim reference on an RTX 4070 Ti SUPER is
279.5 img/s at batch 1 and 506.1 img/s at batch 8.

In [ ]:
import time

for _ in range(20):                                   # warmup
    det.detect(frame)

n = 200
t0 = time.perf_counter()
for _ in range(n):
    det.detect(frame)
b1 = n / (time.perf_counter() - t0)

batch = [frame] * 8                                   # engine was built with --max-batch 8
for _ in range(5):
    det.detect_batch(batch)
t0 = time.perf_counter()
for _ in range(n // 8):
    det.detect_batch(batch)
b8 = (n // 8) * 8 / (time.perf_counter() - t0)

print(f"batch-1: {b1:6.0f} img/s   ({1000 / b1:.2f} ms/frame)")
print(f"batch-8: {b8:6.0f} img/s   ({1000 / b8 * 8:.2f} ms/batch)")

## Where to go next

- **Runtime modes** — GPU decode, frozen buffers, and CUDA Graph requirements are documented
  in [Runtime](https://github.com/PogChamper/dfine-cpp/blob/main/docs/RUNTIME.md).
- **Research presets** — measured accuracy/throughput trade-offs remain separate from the
  default in the [research matrix](https://github.com/PogChamper/dfine-cpp/blob/main/docs/RESEARCH_MATRIX.md).
- **Custom weights** — strict checkpoint export, sidecars, and build commands are covered in
  [Conversion](https://github.com/PogChamper/dfine-cpp/blob/main/docs/CONVERSION.md).